# Lab 1 — Ghost Providers: Build a Directory Agent with Cortex Code
### Triple-S Snow Day · Business Track · 90 minutes

**Audience:** Network Management, Member Services, Compliance, Data Governance, Business & Systems Analysts. **No SQL experience required.**

---

## The business problem

Triple-S publishes a provider directory that members use to find a doctor. Some providers in that directory **no longer work with Triple-S**: their contract ended, their credentialing expired, or their contract has not started yet. But their record is still flagged as `publish = 'Y'`.

When a member calls one of these *ghost providers* they are turned away. That leads to:

* **Member complaints** and a poor experience
* **Appeals and grievances** when care is delayed
* **Regulatory exposure**: CMS audits directory accuracy for Medicare Advantage plans

Today, finding these records means opening a ticket with IT and waiting.

## What you will build today

By the end of this lab you will have:

1. Explored the provider data **by asking Cortex Code in plain language**
2. Discovered why "active provider" is **a business definition, not a column**
3. Built a **semantic view**: the governed business vocabulary that AI uses to answer questions
4. Watched a question fail, then **fixed the semantic view, not the prompt**
5. Created a **Cortex Agent** your team can ask questions like *"Which cardiologists in Bayamón are published but not active?"*
6. Produced the **real count of ghost providers** and a cleanup worklist

> **All data in this lab is synthetic.** Names, NPIs and phone numbers are fake but shaped like real Triple-S data.

## Agenda (90 min)

| Time | Section | What you learn about Cortex Code |
|---|---|---|
| 0–10 | **0. Setup + Cortex Code primer** | How to open Cortex Code and write a good prompt |
| 10–25 | **1. Explore the data** | Ask Cortex Code to explain tables you have never seen |
| 25–35 | **2. Profile for problems** | Ask Cortex Code to find inconsistencies |
| 35–50 | **3. Define "active provider"** | Give Cortex Code *business rules*, not SQL |
| 50–65 | **4. Semantic view + deliberate failure** | Ask Cortex Code for a plan, review it, then fix the model |
| 65–80 | **5. Build the agent** | Let Cortex Code create and test an agent |
| 80–90 | **6. Outcome + wrap-up** | Turn the answer into an action list |

---
# Section 0 — Setup and Cortex Code primer

## What is Cortex Code?

**Cortex Code** is the AI assistant built into Snowflake. It can read your tables, write and run SQL, build semantic views and agents, explain results, and fix its own errors. Think of it as a data analyst sitting next to you who already knows Snowflake.

### How to open it
In this notebook, select the **Cortex Code icon in the lower-right corner** of Snowsight. A chat panel opens next to the notebook. Cortex Code can see the notebook and the objects your role can access.

### The 5 habits of a productive Cortex Code user
You will practice these habits all through the lab:

| # | Habit | Example |
|---|---|---|
| 1 | **Give context**: name the tables and the goal | *"Using the tables in my LAB1 schema, help me find providers that should not be in the directory."* |
| 2 | **Ask for a plan before code** | *"Before writing anything, tell me your plan."* |
| 3 | **State business rules in plain language** | *"A provider is active only if the contract is in force today AND credentialing is valid."* |
| 4 | **Ask it to explain** | *"Explain this query to me like I'm not technical."* |
| 5 | **Verify and iterate** | *"That number looks low. Check if any contracts have status ACTIVE but an end date in the past."* |

> **Throughout the lab** you will see boxes labeled **Cortex Code prompt**. Copy the prompt into the Cortex Code panel, read what it proposes, and approve it. Each prompt is followed by a **reference cell** that produces the same result, so you can always catch up by running that cell.

## 0.1 Set up your account and load the synthetic data

**Everything this lab needs is created by the notebook itself.** You don't need to run anything else. You are `ACCOUNTADMIN` in your own lab account, so the next cell:

| Step | What it does |
|---|---|
| Role | Switches to `ACCOUNTADMIN` |
| Warehouse | Creates and uses `TSS_SNOWDAY_WH` (X-Small, suspends after 60 seconds) |
| Cortex AI | Enables cross-region inference, so every AI model is available |
| Database | Creates `TSS_SNOWDAY.LAB1`, where all your lab objects live |

Every cell after this one uses the variable `fq` (the fully qualified schema name).

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()

# You own this Snowflake account (ACCOUNTADMIN), so the lab sets up everything it needs.
session.sql("USE ROLE ACCOUNTADMIN").collect()

db = "TSS_SNOWDAY"
schema = "LAB1"
fq = f"{db}.{schema}"
wh = "TSS_SNOWDAY_WH"

setup_statements = [
    # Small warehouse for the lab queries, AI functions, Dynamic Tables and the agent
    f"CREATE WAREHOUSE IF NOT EXISTS {wh} WAREHOUSE_SIZE = XSMALL AUTO_SUSPEND = 60 AUTO_RESUME = TRUE",
    f"USE WAREHOUSE {wh}",
    # Let Cortex AI use models from any region if a model is not available in this account's region
    "ALTER ACCOUNT SET CORTEX_ENABLED_CROSS_REGION = 'ANY_REGION'",
    f"CREATE DATABASE IF NOT EXISTS {db}",
    f"CREATE SCHEMA IF NOT EXISTS {fq}",
    f"USE SCHEMA {fq}",
]
for stmt in setup_statements:
    session.sql(stmt).collect()

print(f"Role            : {session.get_current_role()}")
print(f"Your lab schema : {fq}")
print(f"Warehouse       : {wh}")

### Generate the synthetic provider data

The next four cells build four tables that mimic Triple-S provider systems:

| Table | What it represents | Typical system of record |
|---|---|---|
| `PROVIDERS` | One row per provider, including the **directory publish flag** | Provider directory |
| `PROVIDER_CONTRACTS` | The provider's contract: line of business, status, start and end dates | Contracting system |
| `PROVIDER_CREDENTIALS` | Credentialing status and expiration | Credentialing system |
| `DIRECTORY_COMPLAINTS` | Member complaints about the directory | Member Services CRM |

These cells are safe to re-run. They always rebuild the same data.

In [ ]:
LOAD_STATEMENTS = [
    f"""
-- Scenario seed: one row per provider with a hidden "scenario" bucket (0-99)
CREATE OR REPLACE TRANSIENT TABLE {fq}._PROVIDER_GEN AS
SELECT n,
       MOD(ABS(HASH(n, 'scenario')), 100) AS s,
       MOD(ABS(HASH(n, 'd1')), 700)       AS d1,
       MOD(ABS(HASH(n, 'd2')), 365)       AS d2
FROM (SELECT ROW_NUMBER() OVER (ORDER BY SEQ4()) AS n FROM TABLE(GENERATOR(ROWCOUNT => 2000)))
""",
    f"""
CREATE OR REPLACE TABLE {fq}.PROVIDERS AS
SELECT
    n                                                                         AS PROVIDER_ID,
    TO_VARCHAR(1000000000 + MOD(ABS(HASH(n, 'npi')), 899999999))              AS NPI,
    ARRAY_CONSTRUCT('José','María','Luis','Carmen','Carlos','Ana','Jorge','Sofía','Miguel','Isabel','Rafael','Lucía','Juan','Gabriela','Pedro','Valeria','Ángel','Natalia','Héctor','Daniela')[MOD(ABS(HASH(n, 'fn')), 20)]::STRING            AS FIRST_NAME,
    ARRAY_CONSTRUCT('Rivera','Rodríguez','Santiago','Ortiz','Torres','Colón','Vázquez','Pérez','Rosario','Cruz','Martínez','Figueroa','Díaz','Morales','Ramos','González','Meléndez','Negrón','Reyes','Matos')[MOD(ABS(HASH(n, 'ln1')), 20)]::STRING || ' ' ||
    ARRAY_CONSTRUCT('Rivera','Rodríguez','Santiago','Ortiz','Torres','Colón','Vázquez','Pérez','Rosario','Cruz','Martínez','Figueroa','Díaz','Morales','Ramos','González','Meléndez','Negrón','Reyes','Matos')[MOD(ABS(HASH(n, 'ln2')), 20)]::STRING           AS LAST_NAME,
    ARRAY_CONSTRUCT('Family Medicine','Family Medicine','Internal Medicine','Internal Medicine','Pediatrics','Pediatrics','Cardiology','Endocrinology','Orthopedic Surgery','Obstetrics and Gynecology','Dermatology','Psychiatry','Ophthalmology','Gastroenterology','Nephrology','Pulmonology')[MOD(ABS(HASH(n, 'sp')), 16)]::STRING               AS SPECIALTY,
    ARRAY_CONSTRUCT('San Juan','San Juan','San Juan','Bayamón','Bayamón','Carolina','Ponce','Ponce','Caguas','Guaynabo','Mayagüez','Arecibo','Humacao','Fajardo','Aguadilla','Toa Baja','Manatí','Cayey')[MOD(ABS(HASH(n, 'mu')), 18)]::STRING               AS MUNICIPALITY,
    '787-' || TO_VARCHAR(200 + MOD(ABS(HASH(n, 'p1')), 799)) || '-' ||
    LPAD(TO_VARCHAR(MOD(ABS(HASH(n, 'p2')), 10000)), 4, '0')                  AS PHONE,
    IFF(s BETWEEN 70 AND 80, 'N', 'Y')                                        AS PUBLISH_FLAG,
    IFF(MOD(ABS(HASH(n, 'anp')), 100) < 75, 'Y', 'N')                         AS ACCEPTING_NEW_PATIENTS,
    DATEADD(day, -MOD(ABS(HASH(n, 'upd')), 900), CURRENT_DATE())              AS LAST_DIRECTORY_UPDATE
FROM {fq}._PROVIDER_GEN
""",
    f"""
CREATE OR REPLACE TABLE {fq}.PROVIDER_CONTRACTS AS
SELECT
    'C-' || LPAD(TO_VARCHAR(n), 6, '0') AS CONTRACT_ID,
    n AS PROVIDER_ID,
    CASE WHEN MOD(ABS(HASH(n, 'lob')), 100) < 50 THEN 'Medicare Advantage'
         WHEN MOD(ABS(HASH(n, 'lob')), 100) < 80 THEN 'Commercial'
         ELSE 'Vital (Medicaid)' END AS LINE_OF_BUSINESS,
    CASE WHEN s BETWEEN 70 AND 77 OR s BETWEEN 81 AND 84 THEN 'TERMINATED'
         WHEN s BETWEEN 78 AND 80 THEN 'PENDING'
         ELSE 'ACTIVE' END AS CONTRACT_STATUS,
    CASE WHEN s BETWEEN 78 AND 80 THEN DATEADD(day, 10 + MOD(d2, 60), CURRENT_DATE())
         WHEN s BETWEEN 91 AND 92 THEN DATEADD(day,  5 + MOD(d2, 90), CURRENT_DATE())
         WHEN s BETWEEN 70 AND 77 OR s BETWEEN 81 AND 88 THEN DATEADD(day, -(d1 + 400), CURRENT_DATE())
         ELSE DATEADD(day, -(d1 + 30), CURRENT_DATE()) END AS EFFECTIVE_DATE,
    CASE WHEN s BETWEEN 70 AND 77 OR s BETWEEN 81 AND 84 THEN DATEADD(day, -(d2 + 1), CURRENT_DATE())
         WHEN s BETWEEN 85 AND 88 THEN DATEADD(day, -(MOD(d2, 300) + 1), CURRENT_DATE())
         WHEN s BETWEEN 93 AND 99 THEN DATEADD(day, 1 + MOD(d2, 90), CURRENT_DATE())
         WHEN s BETWEEN 78 AND 80 OR s BETWEEN 91 AND 92 THEN NULL
         WHEN MOD(n, 3) = 0 THEN NULL
         ELSE DATEADD(day, 120 + d2, CURRENT_DATE()) END AS TERMINATION_DATE
FROM {fq}._PROVIDER_GEN
""",
    f"""
CREATE OR REPLACE TABLE {fq}.PROVIDER_CREDENTIALS AS
SELECT
    n AS PROVIDER_ID,
    CASE WHEN s BETWEEN 89 AND 90 THEN 'EXPIRED'
         WHEN s BETWEEN 78 AND 80 THEN 'IN_REVIEW'
         ELSE 'ACTIVE' END AS CREDENTIAL_STATUS,
    DATEADD(day, -(d1 + 200), CURRENT_DATE()) AS LAST_CREDENTIALED_DATE,
    CASE WHEN s BETWEEN 89 AND 90 THEN DATEADD(day, -(MOD(d2, 200) + 1), CURRENT_DATE())
         ELSE DATEADD(day, 30 + d1, CURRENT_DATE()) END AS CREDENTIAL_EXPIRATION_DATE
FROM {fq}._PROVIDER_GEN
""",
    f"""
CREATE OR REPLACE TABLE {fq}.DIRECTORY_COMPLAINTS AS
WITH c AS (
    SELECT ROW_NUMBER() OVER (ORDER BY SEQ4()) AS k FROM TABLE(GENERATOR(ROWCOUNT => 4000))
), p AS (
    SELECT k, 1 + MOD(ABS(HASH(k, 'prov')), 2000) AS pid, MOD(ABS(HASH(k, 'keep')), 100) AS keep FROM c
)
SELECT
    'CMP-' || LPAD(TO_VARCHAR(k), 6, '0') AS COMPLAINT_ID,
    p.pid AS PROVIDER_ID,
    DATEADD(day, -MOD(ABS(HASH(k, 'dt')), 180), CURRENT_DATE()) AS COMPLAINT_DATE,
    ARRAY_CONSTRUCT('Office said provider no longer accepts Triple-S',
                    'Provider is not taking appointments from plan members',
                    'Provider retired or moved',
                    'Phone number in directory is wrong',
                    'Member was billed as out of network')[MOD(ABS(HASH(k, 'r')), 5)]::STRING AS COMPLAINT_REASON,
    ARRAY_CONSTRUCT('Call Center', 'Call Center', 'Member Portal', 'Mobile App')[MOD(ABS(HASH(k, 'ch')), 4)]::STRING AS CHANNEL
FROM p
JOIN {fq}._PROVIDER_GEN g ON g.n = p.pid
WHERE (g.s BETWEEN 81 AND 92 AND p.keep < 45) OR p.keep < 2
""",
    f"""
DROP TABLE IF EXISTS {fq}._PROVIDER_GEN
"""
]

for i, stmt in enumerate(LOAD_STATEMENTS, 1):
    session.sql(stmt).collect()
    print(f"Step {i}/{len(LOAD_STATEMENTS)} done")
print("Synthetic provider data loaded.")

In [ ]:
%%sql -r load_check
SELECT 'PROVIDERS' AS TABLE_NAME, COUNT(*) AS ROW_COUNT FROM {{fq}}.PROVIDERS
UNION ALL SELECT 'PROVIDER_CONTRACTS', COUNT(*) FROM {{fq}}.PROVIDER_CONTRACTS
UNION ALL SELECT 'PROVIDER_CREDENTIALS', COUNT(*) FROM {{fq}}.PROVIDER_CREDENTIALS
UNION ALL SELECT 'DIRECTORY_COMPLAINTS', COUNT(*) FROM {{fq}}.DIRECTORY_COMPLAINTS

---
# Section 1 — Explore the data with Cortex Code (15 min)

You have just received four tables you have never seen. The traditional path is to ask IT for a data dictionary. **Instead, ask Cortex Code.**

> **Cortex Code prompt**
> ```
> I'm a business analyst on the Triple-S Network team. Look at the tables in my
> schema TSS_SNOWDAY.LAB1. For each table, explain in plain English what
> it represents, what one row means, and how the tables connect to each other.
> Then show me 5 sample providers with their contract and credential information.
> ```

**What to notice:** Cortex Code reads the table metadata, infers the join key (`PROVIDER_ID`), and explains it in business language. You did not write any SQL.

**Try a follow-up:** *"Which columns would matter most for deciding whether a provider should appear in the member directory?"*

**Reference cell:** the same result Cortex Code should produce. Run it if you want to catch up.

In [ ]:
%%sql -r sample_providers
SELECT p.PROVIDER_ID, p.FIRST_NAME, p.LAST_NAME, p.SPECIALTY, p.MUNICIPALITY, p.PUBLISH_FLAG,
       c.LINE_OF_BUSINESS, c.CONTRACT_STATUS, c.EFFECTIVE_DATE, c.TERMINATION_DATE,
       cr.CREDENTIAL_STATUS, cr.CREDENTIAL_EXPIRATION_DATE
FROM {{fq}}.PROVIDERS p
JOIN {{fq}}.PROVIDER_CONTRACTS c  ON c.PROVIDER_ID  = p.PROVIDER_ID
JOIN {{fq}}.PROVIDER_CREDENTIALS cr ON cr.PROVIDER_ID = p.PROVIDER_ID
ORDER BY p.PROVIDER_ID
LIMIT 5

---
# Section 2 — Profile the data for problems (10 min)

Now ask Cortex Code to act like a **data quality analyst**. Notice that the prompt states a goal, not a query.

> **Cortex Code prompt**
> ```
> Our member directory should only show providers that members can actually see.
> Profile these tables and find inconsistencies between the PUBLISH_FLAG in
> PROVIDERS and the contract and credential data. Show me a summary table of
> published vs. not published providers by contract status.
> ```

**What to notice:** Cortex Code usually finds the *obvious* problem first: providers that are published but whose contract status is `TERMINATED`. Write that number down.

In [ ]:
%%sql -r publish_vs_status
SELECT p.PUBLISH_FLAG, c.CONTRACT_STATUS, COUNT(*) AS PROVIDERS
FROM {{fq}}.PROVIDERS p
JOIN {{fq}}.PROVIDER_CONTRACTS c ON c.PROVIDER_ID = p.PROVIDER_ID
GROUP BY 1, 2
ORDER BY 1 DESC, 3 DESC

### Checkpoint
If someone asked *"How many ghost providers do we have?"*, most people would answer with the **PUBLISH_FLAG = 'Y' and CONTRACT_STATUS = 'TERMINATED'** number above.

**That answer is wrong.** Let's see why.

---
# Section 3 — Define "active provider": it's a business rule, not a column (15 min)

This is the **core lesson** of the lab.

There is no column called `IS_ACTIVE`. Whether a provider can be shown to members depends on **several fields in three different systems** plus **a date window**:

| # | Rule | Where it lives |
|---|---|---|
| 1 | The provider is flagged to publish | `PROVIDERS.PUBLISH_FLAG = 'Y'` |
| 2 | The contract status is `ACTIVE` | `PROVIDER_CONTRACTS.CONTRACT_STATUS` |
| 3 | **Today** falls inside the contract window: started, and not yet ended | `EFFECTIVE_DATE <= today` and (`TERMINATION_DATE` is empty or `>= today`) |
| 4 | Credentialing is active and not expired | `PROVIDER_CREDENTIALS` |

**Only the business can define this.** IT can tell you what a column contains. Only the Network team knows that a contract with status `ACTIVE` but an end date last month is **not** active. That happens when the contracting system was never updated.

> **Cortex Code prompt** (notice: business rules, no SQL)
> ```
> A provider is ACTIVE for the member directory only if ALL of these are true:
> 1) contract status is ACTIVE, 2) the contract effective date is on or before today,
> 3) the termination date is empty or on or after today, and 4) the credential status
> is ACTIVE and the credential has not expired.
> A GHOST provider is one that is published (PUBLISH_FLAG = 'Y') but NOT active.
> Count ghost providers and break them down by the reason they fail.
> Compare that total to the naive count of published + TERMINATED.
> ```

**Reference cell:** builds a single directory view that joins the three systems. It holds no business logic yet, just the raw facts.

In [ ]:
%%sql -r create_directory_view
CREATE OR REPLACE VIEW {{fq}}.PROVIDER_DIRECTORY
COMMENT = 'One row per provider: directory, contract and credential facts joined. No business logic.'
AS
SELECT p.PROVIDER_ID, p.NPI, p.FIRST_NAME, p.LAST_NAME, p.SPECIALTY, p.MUNICIPALITY, p.PHONE,
       p.PUBLISH_FLAG, p.ACCEPTING_NEW_PATIENTS, p.LAST_DIRECTORY_UPDATE,
       c.CONTRACT_ID, c.LINE_OF_BUSINESS, c.CONTRACT_STATUS, c.EFFECTIVE_DATE, c.TERMINATION_DATE,
       cr.CREDENTIAL_STATUS, cr.CREDENTIAL_EXPIRATION_DATE
FROM {{fq}}.PROVIDERS p
JOIN {{fq}}.PROVIDER_CONTRACTS c   ON c.PROVIDER_ID  = p.PROVIDER_ID
JOIN {{fq}}.PROVIDER_CREDENTIALS cr ON cr.PROVIDER_ID = p.PROVIDER_ID

In [ ]:
%%sql -r naive_vs_business
-- The "aha" moment: naive count vs. the business definition
SELECT
    COUNT_IF(PUBLISH_FLAG = 'Y' AND CONTRACT_STATUS = 'TERMINATED') AS NAIVE_GHOST_COUNT,
    COUNT_IF(PUBLISH_FLAG = 'Y' AND NOT (
            CONTRACT_STATUS = 'ACTIVE'
        AND EFFECTIVE_DATE <= CURRENT_DATE()
        AND (TERMINATION_DATE IS NULL OR TERMINATION_DATE >= CURRENT_DATE())
        AND CREDENTIAL_STATUS = 'ACTIVE'
        AND CREDENTIAL_EXPIRATION_DATE >= CURRENT_DATE()
    )) AS BUSINESS_RULE_GHOST_COUNT
FROM {{fq}}.PROVIDER_DIRECTORY

In [ ]:
%%sql -r ghost_reasons
SELECT
    CASE
        WHEN CONTRACT_STATUS = 'TERMINATED'                                   THEN '1. Contract terminated'
        WHEN CONTRACT_STATUS = 'ACTIVE' AND TERMINATION_DATE < CURRENT_DATE()  THEN '2. Status ACTIVE but end date has passed (stale record)'
        WHEN EFFECTIVE_DATE > CURRENT_DATE()                                   THEN '3. Contract has not started yet'
        WHEN CREDENTIAL_STATUS <> 'ACTIVE' OR CREDENTIAL_EXPIRATION_DATE < CURRENT_DATE() THEN '4. Credential expired'
        ELSE 'Other'
    END AS GHOST_REASON,
    COUNT(*) AS GHOST_PROVIDERS
FROM {{fq}}.PROVIDER_DIRECTORY
WHERE PUBLISH_FLAG = 'Y' AND NOT (
        CONTRACT_STATUS = 'ACTIVE'
    AND EFFECTIVE_DATE <= CURRENT_DATE()
    AND (TERMINATION_DATE IS NULL OR TERMINATION_DATE >= CURRENT_DATE())
    AND CREDENTIAL_STATUS = 'ACTIVE'
    AND CREDENTIAL_EXPIRATION_DATE >= CURRENT_DATE())
GROUP BY 1
ORDER BY 1

### What just happened
The naive count missed **about two out of every three ghost providers**. The biggest hidden group is reason #2: contracts that still say `ACTIVE` but whose end date has already passed. **No column tells you that.** Only the business rule does.

> **Ask Cortex Code to explain:** *"Explain the difference between these two counts in two sentences I could send to my director."*

---
# Section 4 — Semantic view: teach the AI your business vocabulary (15 min)

A **semantic view** is a Snowflake object that describes your data in business terms: what a "provider" is, what "active" means, which synonyms people use (*pueblo*, *municipio*, *town*). Cortex Agents and Cortex Analyst read the semantic view to turn a natural-language question into correct SQL.

**Key idea:** if the semantic view has the wrong definition, *every* AI answer built on it will be wrong, however well you write the prompt.

## 4.1 The semantic view IT built last year (version 1)

Imagine IT built this semantic view a year ago. It looks reasonable, but it defines *active provider* as **just** `CONTRACT_STATUS = 'ACTIVE'`.

> **Cortex Code prompt** (practice habit #2: ask for a plan first)
> ```
> I want to create a semantic view called PROVIDER_DIRECTORY_SV on the view
> PROVIDER_DIRECTORY in my schema so business users can ask questions in English
> or Spanish. Before creating anything, show me your plan: which dimensions,
> metrics and synonyms you would include.
> ```
> Review the plan. Did Cortex Code pick up the business rule from Section 3? Then run the reference cell below, which deliberately reproduces the *IT version* with the naive definition.

In [ ]:
%%sql -r semantic_view_v1
CREATE OR REPLACE SEMANTIC VIEW {{fq}}.PROVIDER_DIRECTORY_SV
  TABLES (
    directory AS {{fq}}.PROVIDER_DIRECTORY
      PRIMARY KEY (PROVIDER_ID)
      COMMENT = 'One row per provider in the Triple-S provider directory, with contract and credential facts'
  )
  DIMENSIONS (
    directory.provider_name AS FIRST_NAME || ' ' || LAST_NAME
      WITH SYNONYMS = ('doctor', 'physician', 'proveedor', 'médico')
      COMMENT = 'Full name of the provider',
    directory.specialty AS SPECIALTY
      WITH SYNONYMS = ('especialidad')
      COMMENT = 'Medical specialty, for example Cardiology',
    directory.municipality AS MUNICIPALITY
      WITH SYNONYMS = ('town', 'city', 'pueblo', 'municipio')
      COMMENT = 'Puerto Rico municipality of the practice',
    directory.line_of_business AS LINE_OF_BUSINESS
      WITH SYNONYMS = ('product', 'línea de negocio')
      COMMENT = 'Medicare Advantage, Commercial or Vital (Medicaid)',
    directory.is_published AS PUBLISH_FLAG = 'Y'
      WITH SYNONYMS = ('in the directory', 'publicado')
      COMMENT = 'TRUE when the provider is shown to members in the directory',
    directory.contract_status AS CONTRACT_STATUS
      COMMENT = 'Status recorded in the contracting system',
    directory.is_active_provider AS CONTRACT_STATUS = 'ACTIVE'
      WITH SYNONYMS = ('active', 'activo', 'in network')
      COMMENT = 'TRUE when the provider is active'
  )
  METRICS (
    directory.provider_count AS COUNT(PROVIDER_ID)
      WITH SYNONYMS = ('number of providers', 'cantidad de proveedores')
      COMMENT = 'Number of providers'
  )
  COMMENT = 'Triple-S provider directory (v1, IT definition)'

## 4.2 Deliberate failure: ask a question

A Network analyst asks: **"How many providers are published but not active?"**

The next cell asks the semantic view exactly that question, the same way an agent would.

In [ ]:
%%sql -r v1_answer
SELECT *
FROM SEMANTIC_VIEW(
    {{fq}}.PROVIDER_DIRECTORY_SV
    DIMENSIONS directory.is_published, directory.is_active_provider
    METRICS directory.provider_count
)
ORDER BY 1 DESC, 2

### Something is wrong
Find the row where `IS_PUBLISHED = TRUE` and `IS_ACTIVE_PROVIDER = FALSE`. It matches the **naive count** from Section 2, not the **business-rule count** from Section 3.

The question was right, the SQL ran without errors, and **the answer is still wrong**. Rewording the question won't help: *"published but inactive"*, *"published and not in network"* and *"¿publicados pero no activos?"* all give the same wrong number, because the definition underneath is wrong.

> ### The lesson: fix the semantic view, not the prompt.

## 4.3 Fix the semantic view with Cortex Code (version 2)

> **Cortex Code prompt** (practice habit #3: state business rules in plain language)
> ```
> The semantic view PROVIDER_DIRECTORY_SV defines is_active_provider as only
> CONTRACT_STATUS = 'ACTIVE'. That is wrong for the business. Update it so that:
> - is_active_provider = contract status ACTIVE, effective date <= today,
>   termination date empty or >= today, credential ACTIVE and not expired
> - add is_ghost_provider = published AND NOT active
> - add ghost_reason with the four reasons we found in Section 3
> - add is_contract_expiring_next_90_days
> - add a metric ghost_provider_count
> Keep the Spanish synonyms. Show me the DDL before you run it.
> ```

In [ ]:
%%sql -r semantic_view_v2
CREATE OR REPLACE SEMANTIC VIEW {{fq}}.PROVIDER_DIRECTORY_SV
  TABLES (
    directory AS {{fq}}.PROVIDER_DIRECTORY
      PRIMARY KEY (PROVIDER_ID)
      COMMENT = 'One row per provider in the Triple-S provider directory, with contract and credential facts'
  )
  FACTS (
    directory.days_until_contract_end AS DATEDIFF(day, CURRENT_DATE(), TERMINATION_DATE)
      COMMENT = 'Days from today until the contract termination date (negative = already ended)'
  )
  DIMENSIONS (
    directory.provider_id AS PROVIDER_ID
      COMMENT = 'Internal provider identifier',
    directory.npi AS NPI
      COMMENT = 'National Provider Identifier',
    directory.provider_name AS FIRST_NAME || ' ' || LAST_NAME
      WITH SYNONYMS = ('doctor', 'physician', 'proveedor', 'médico')
      COMMENT = 'Full name of the provider',
    directory.specialty AS SPECIALTY
      WITH SYNONYMS = ('especialidad')
      COMMENT = 'Medical specialty. Values include Cardiology, Pediatrics, Family Medicine, Internal Medicine, Endocrinology',
    directory.municipality AS MUNICIPALITY
      WITH SYNONYMS = ('town', 'city', 'pueblo', 'municipio')
      COMMENT = 'Puerto Rico municipality of the practice, for example Bayamón, San Juan, Ponce',
    directory.line_of_business AS LINE_OF_BUSINESS
      WITH SYNONYMS = ('product', 'línea de negocio')
      COMMENT = 'Medicare Advantage, Commercial or Vital (Medicaid)',
    directory.phone AS PHONE
      COMMENT = 'Phone number published in the directory',
    directory.is_published AS PUBLISH_FLAG = 'Y'
      WITH SYNONYMS = ('in the directory', 'publicado')
      COMMENT = 'TRUE when the provider is shown to members in the directory',
    directory.contract_status AS CONTRACT_STATUS
      COMMENT = 'Raw status recorded in the contracting system. Do NOT use alone to decide if a provider is active',
    directory.contract_termination_date AS TERMINATION_DATE
      WITH SYNONYMS = ('contract end date', 'fecha de terminación')
      COMMENT = 'Date the contract ends. Empty means no end date',
    directory.is_active_provider AS (
        CONTRACT_STATUS = 'ACTIVE'
        AND EFFECTIVE_DATE <= CURRENT_DATE()
        AND (TERMINATION_DATE IS NULL OR TERMINATION_DATE >= CURRENT_DATE())
        AND CREDENTIAL_STATUS = 'ACTIVE'
        AND CREDENTIAL_EXPIRATION_DATE >= CURRENT_DATE()
      )
      WITH SYNONYMS = ('active', 'activo', 'in network', 'participating')
      COMMENT = 'Business definition (Network Management): contract ACTIVE and in force today AND credential ACTIVE and not expired',
    directory.is_ghost_provider AS (
        PUBLISH_FLAG = 'Y' AND NOT (CONTRACT_STATUS = 'ACTIVE'
        AND EFFECTIVE_DATE <= CURRENT_DATE()
        AND (TERMINATION_DATE IS NULL OR TERMINATION_DATE >= CURRENT_DATE())
        AND CREDENTIAL_STATUS = 'ACTIVE'
        AND CREDENTIAL_EXPIRATION_DATE >= CURRENT_DATE())
      )
      WITH SYNONYMS = ('ghost', 'fantasma', 'proveedor fantasma', 'should not be published')
      COMMENT = 'TRUE when a provider is published in the directory but is NOT active. These must be cleaned up',
    directory.ghost_reason AS CASE
        WHEN NOT (PUBLISH_FLAG = 'Y' AND NOT (CONTRACT_STATUS = 'ACTIVE'
        AND EFFECTIVE_DATE <= CURRENT_DATE()
        AND (TERMINATION_DATE IS NULL OR TERMINATION_DATE >= CURRENT_DATE())
        AND CREDENTIAL_STATUS = 'ACTIVE'
        AND CREDENTIAL_EXPIRATION_DATE >= CURRENT_DATE())) THEN NULL
        WHEN CONTRACT_STATUS = 'TERMINATED' THEN 'Contract terminated'
        WHEN CONTRACT_STATUS = 'ACTIVE' AND TERMINATION_DATE < CURRENT_DATE() THEN 'Status ACTIVE but end date has passed'
        WHEN EFFECTIVE_DATE > CURRENT_DATE() THEN 'Contract has not started yet'
        ELSE 'Credential expired' END
      WITH SYNONYMS = ('razón', 'reason')
      COMMENT = 'Why a ghost provider fails the active definition',
    directory.is_contract_expiring_next_90_days AS (
        CONTRACT_STATUS = 'ACTIVE'
        AND TERMINATION_DATE BETWEEN CURRENT_DATE() AND DATEADD(day, 90, CURRENT_DATE())
      )
      WITH SYNONYMS = ('expiring next quarter', 'vence el próximo trimestre', 'por vencer')
      COMMENT = 'TRUE when an active contract ends in the next 90 days'
  )
  METRICS (
    directory.provider_count AS COUNT(PROVIDER_ID)
      WITH SYNONYMS = ('number of providers', 'cantidad de proveedores')
      COMMENT = 'Number of providers',
    directory.ghost_provider_count AS COUNT_IF(PUBLISH_FLAG = 'Y' AND NOT (CONTRACT_STATUS = 'ACTIVE'
        AND EFFECTIVE_DATE <= CURRENT_DATE()
        AND (TERMINATION_DATE IS NULL OR TERMINATION_DATE >= CURRENT_DATE())
        AND CREDENTIAL_STATUS = 'ACTIVE'
        AND CREDENTIAL_EXPIRATION_DATE >= CURRENT_DATE()))
      WITH SYNONYMS = ('ghost providers', 'proveedores fantasma')
      COMMENT = 'Number of providers published in the directory but not active'
  )
  COMMENT = 'Triple-S provider directory (v2, business definition owned by Network Management)'

Ask the **same question** again, with no change to the question:

In [ ]:
%%sql -r v2_answer
SELECT *
FROM SEMANTIC_VIEW(
    {{fq}}.PROVIDER_DIRECTORY_SV
    DIMENSIONS directory.is_published, directory.is_active_provider
    METRICS directory.provider_count
)
ORDER BY 1 DESC, 2

In [ ]:
%%sql -r v2_ghosts_by_reason
SELECT *
FROM SEMANTIC_VIEW(
    {{fq}}.PROVIDER_DIRECTORY_SV
    DIMENSIONS directory.ghost_reason
    METRICS directory.ghost_provider_count
    WHERE directory.is_ghost_provider
)
ORDER BY 2 DESC

The count now matches the business definition. **Same question, correct answer**, because the business owns the definition in the semantic view.

> **Governance moment:** the business rule now lives in **one governed object**. Every agent, dashboard and analyst that uses `PROVIDER_DIRECTORY_SV` gets the same answer. Ask Cortex Code: *"Write a one-paragraph data dictionary entry for is_active_provider that I can share with Compliance."*

---
# Section 5 — Build the Ghost Provider agent (15 min)

A **Cortex Agent** wraps the semantic view in a conversational assistant your team can use in **CoWork** (Snowflake's AI workspace) without opening a notebook.

> **Cortex Code prompt**
> ```
> Create a Cortex Agent called GHOST_PROVIDER_AGENT in my schema that uses the
> semantic view PROVIDER_DIRECTORY_SV through a Cortex Analyst tool. It should
> answer in the same language as the question (Spanish or English), and add
> sample questions about ghost providers, cardiology in Bayamón, and contracts
> expiring next quarter. Show me the specification before creating it.
> ```

In [ ]:
%%sql -r create_agent
CREATE OR REPLACE AGENT {{fq}}.GHOST_PROVIDER_AGENT
  COMMENT = 'Triple-S Snow Day: helps Network Management find and clean up ghost providers'
  PROFILE = '{"display_name": "Triple-S Directory Accuracy Assistant"}'
  FROM SPECIFICATION
  $$
  models:
    orchestration: auto

  instructions:
    response: "You help the Triple-S Network Management team keep the provider directory accurate for members and for CMS directory audits. Always answer in the same language the user writes in (Spanish or English). State the number of providers first. When you list providers, include name, NPI, specialty, municipality, line of business and, if applicable, the ghost reason."
    orchestration: "Use the directory_analyst tool for every question about providers, contracts, credentials, the directory or ghost providers. A provider is active ONLY as defined by is_active_provider in the semantic view; never use contract_status alone."
    sample_questions:
      - question: "¿Qué proveedores están publicados pero tienen el contrato terminado?"
      - question: "Cardiología en Bayamón: ¿cuántos proveedores fantasma hay?"
      - question: "¿Qué contratos vencen el próximo trimestre?"
      - question: "How many ghost providers do we have by line of business?"

  tools:
    - tool_spec:
        type: "cortex_analyst_text_to_sql"
        name: "directory_analyst"
        description: "Answers questions about the Triple-S provider directory: providers, specialties, municipalities, lines of business, contract status and dates, credentialing, active providers and ghost providers (published but not active)."

  tool_resources:
    directory_analyst:
      semantic_view: "{{fq}}.PROVIDER_DIRECTORY_SV"
      execution_environment:
        type: "warehouse"
        warehouse: "{{wh}}"
  $$

In [ ]:
%%sql -r describe_agent
DESCRIBE AGENT {{fq}}.GHOST_PROVIDER_AGENT

## 5.1 Make the agent visible in CoWork

Some accounts have a **CoWork object** (`SNOWFLAKE_INTELLIGENCE_OBJECT_DEFAULT`) that holds the curated list of agents users see. If it exists, an agent shows up in CoWork **only after it is added to that object**. Without this step, the agent works in Snowsight but members of your team won't find it in CoWork.

The next cell checks whether your account has that object and, if it does, adds your agent to it.

> **Cortex Code prompt:** *"Make GHOST_PROVIDER_AGENT visible in CoWork and give me the link to open it."*

In [ ]:
agent_fqn = f"{fq}.GHOST_PROVIDER_AGENT"
si_objects = session.sql("SHOW SNOWFLAKE INTELLIGENCES").collect()

if not si_objects:
    print("No CoWork object in this account: agents appear in CoWork automatically.")
else:
    si_name = si_objects[0]["name"]
    try:
        session.sql(f"ALTER SNOWFLAKE INTELLIGENCE {si_name} ADD AGENT {agent_fqn}").collect()
        print(f"Added {agent_fqn} to CoWork object {si_name}.")
    except Exception as e:
        if "already" in str(e).lower():
            print(f"{agent_fqn} is already in CoWork object {si_name}.")
        else:
            print(f"Could not add the agent to {si_name}. Make sure the setup cell ran as ACCOUNTADMIN.\n{e}")

print("Open CoWork at https://ai.snowflake.com and look for 'Triple-S Directory Accuracy Assistant'.")

## 5.2 Test your agent

1. Open **CoWork** (https://ai.snowflake.com) and select **Triple-S Directory Accuracy Assistant**. You can also test it in Snowsight under **AI & ML » Agents**.
2. Ask these questions **in Spanish**:
   * *¿Qué proveedores están publicados pero tienen el contrato terminado?*
   * *Cardiología en Bayamón: ¿hay proveedores fantasma?*
   * *¿Qué contratos vencen el próximo trimestre?*
   * *¿Cuántos proveedores fantasma tenemos por línea de negocio y por razón?*
3. Expand the agent's reasoning to **see the SQL it generated**. Check that it uses `is_ghost_provider` / `is_active_provider`.

**Discussion:** the first question says *"contrato terminado"*. Should the agent count only `TERMINATED`, or every contract that has ended, including the stale `ACTIVE` ones? This is exactly the kind of business decision your team now owns.

### Verify the agent's answers
The cells below answer the same questions directly against the semantic view, so you can compare them with the agent.

In [ ]:
%%sql -r q_cardiology_bayamon
-- Cardiología en Bayamón
SELECT *
FROM SEMANTIC_VIEW(
    {{fq}}.PROVIDER_DIRECTORY_SV
    DIMENSIONS directory.provider_name, directory.npi, directory.line_of_business,
               directory.is_active_provider, directory.ghost_reason
    WHERE directory.specialty = 'Cardiology' AND directory.municipality = 'Bayamón' AND directory.is_published
)
ORDER BY 4, 1

In [ ]:
%%sql -r q_expiring_next_quarter
-- ¿Qué contratos vencen el próximo trimestre? (next 90 days)
SELECT *
FROM SEMANTIC_VIEW(
    {{fq}}.PROVIDER_DIRECTORY_SV
    DIMENSIONS directory.line_of_business
    METRICS directory.provider_count
    WHERE directory.is_contract_expiring_next_90_days
)
ORDER BY 2 DESC

---
# Section 6 — From answer to action (10 min)

An answer only matters if someone acts on it. Ask Cortex Code to turn the result into a **cleanup worklist** and to show the **member impact**.

> **Cortex Code prompt**
> ```
> Create a table GHOST_PROVIDER_WORKLIST with every ghost provider, the reason,
> and the number of member complaints in DIRECTORY_COMPLAINTS in the last 180 days.
> Sort by complaints, highest first. Then chart ghost providers by municipality,
> colored by reason.
> ```

In [ ]:
%%sql -r create_worklist
CREATE OR REPLACE TABLE {{fq}}.GHOST_PROVIDER_WORKLIST AS
SELECT g.*, COALESCE(c.COMPLAINTS_LAST_180_DAYS, 0) AS COMPLAINTS_LAST_180_DAYS
FROM SEMANTIC_VIEW(
        {{fq}}.PROVIDER_DIRECTORY_SV
        DIMENSIONS directory.provider_id, directory.npi, directory.provider_name, directory.specialty,
                   directory.municipality, directory.line_of_business, directory.phone, directory.ghost_reason
        WHERE directory.is_ghost_provider
     ) g
LEFT JOIN (
    SELECT PROVIDER_ID, COUNT(*) AS COMPLAINTS_LAST_180_DAYS
    FROM {{fq}}.DIRECTORY_COMPLAINTS
    WHERE COMPLAINT_DATE >= DATEADD(day, -180, CURRENT_DATE())
    GROUP BY PROVIDER_ID
) c ON c.PROVIDER_ID = g.provider_id

In [ ]:
%%sql -r worklist_top
SELECT * FROM {{fq}}.GHOST_PROVIDER_WORKLIST
ORDER BY COMPLAINTS_LAST_180_DAYS DESC, PROVIDER_NAME
LIMIT 15

In [ ]:
%%sql -r impact_summary
SELECT
    (SELECT COUNT(*) FROM {{fq}}.GHOST_PROVIDER_WORKLIST)                               AS GHOST_PROVIDERS,
    (SELECT SUM(COMPLAINTS_LAST_180_DAYS) FROM {{fq}}.GHOST_PROVIDER_WORKLIST)          AS COMPLAINTS_FROM_GHOSTS,
    (SELECT COUNT(*) FROM {{fq}}.DIRECTORY_COMPLAINTS
      WHERE COMPLAINT_DATE >= DATEADD(day, -180, CURRENT_DATE()))                     AS ALL_DIRECTORY_COMPLAINTS,
    ROUND(100 * COMPLAINTS_FROM_GHOSTS / NULLIF(ALL_DIRECTORY_COMPLAINTS, 0), 1)         AS PCT_COMPLAINTS_FROM_GHOSTS

In [ ]:
%%sql -r ghosts_by_municipality
SELECT MUNICIPALITY, GHOST_REASON, COUNT(*) AS GHOST_PROVIDERS
FROM {{fq}}.GHOST_PROVIDER_WORKLIST
GROUP BY 1, 2

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = ghosts_by_municipality.to_pandas() if not isinstance(ghosts_by_municipality, pd.DataFrame) else ghosts_by_municipality.copy()
pivot = df.pivot_table(index="MUNICIPALITY", columns="GHOST_REASON", values="GHOST_PROVIDERS", aggfunc="sum", fill_value=0)
pivot = pivot.loc[pivot.sum(axis=1).sort_values().index]

ax = pivot.plot(kind="barh", stacked=True, figsize=(10, 7))
ax.set_title("Ghost providers by municipality and reason")
ax.set_xlabel("Ghost providers")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

---
# Wrap-up

## What you accomplished in 90 minutes
* Found the **real number of ghost providers**, about **3x** the naive count, with no IT ticket
* Turned a business rule into a **governed semantic view** that every AI tool can reuse
* Built an **agent** your team can ask in Spanish
* Produced a **prioritized cleanup worklist** tied to member complaints

## The Cortex Code habits you practiced
| Habit | Where you used it |
|---|---|
| Give context | Section 1: described your role and your schema |
| Ask for a plan first | Section 4.1: reviewed the semantic view plan before creating it |
| State business rules, not SQL | Sections 3 and 4.3 |
| Ask it to explain | Section 3: explanation for your director |
| Verify and iterate | Section 4.2: noticed a wrong number and fixed the model |

## Take-home challenges (try them with Cortex Code)
1. *"Add a dimension for providers not accepting new patients and ask the agent which published cardiologists accept new patients in Ponce."*
2. *"Add DIRECTORY_COMPLAINTS to the semantic view so the agent can answer which ghost providers generate the most complaints."*
3. *"Schedule a weekly task that refreshes GHOST_PROVIDER_WORKLIST and emails a summary to the Network team."*

> **Remember:** the AI is only as good as the business definitions you give it. **Fix the semantic view, not the prompt.**

---
## Teardown (run only after the lab)
This cell is commented out on purpose. Uncomment it and run it to remove **your** lab objects.

In [ ]:
# session.sql(f"DROP AGENT IF EXISTS {fq}.GHOST_PROVIDER_AGENT").collect()
# session.sql(f"DROP SCHEMA IF EXISTS {fq}").collect()
# session.sql(f"DROP WAREHOUSE IF EXISTS {wh}").collect()   # only if you are not running the other lab
# print(f"Dropped {fq}")